# DATA 298B Four-Model AML Evaluation

This notebook runs the fast, no-tool comparison for Ministral, Phi-4, Qwen, and Gemma. It does not train models and does not keep all four models in GPU memory at once.

In [ ]:
# ---- Edit only this cell if your Drive paths differ ----
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-backend'
HELDOUT_DATASET = '/content/drive/MyDrive/DATA298B/heldout/aml_heldout.jsonl'
ADAPTER_ROOT = '/content/drive/MyDrive/DATA298B/adapters'
OUTPUT_ROOT = '/content/drive/MyDrive/DATA298B/final_benchmark'
LEGITIMATE_CASES = 10
SUSPICIOUS_CASES = 40
MAX_TOKENS = 700
print('held-out dataset:', HELDOUT_DATASET)
print('output:', OUTPUT_ROOT)

In [ ]:
# Mount Drive and pull the latest evaluation code.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
REPO_DIR = Path('/content/Graph-Based-Fraud-Intelligence-Platform')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repository ready:', REPO_DIR)

In [ ]:
# Install evaluation dependencies and verify GPU.
%pip install -q -r requirements.txt
%pip install -q accelerate bitsandbytes sentencepiece huggingface_hub
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before starting evaluation.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# Secure Hugging Face login.
from huggingface_hub import login
login(add_to_git_credential=False)

In [ ]:
# Verify held-out cases, adapters, and expected separation.
import json
from pathlib import Path
heldout = Path(HELDOUT_DATASET)
adapter_root = Path(ADAPTER_ROOT)
if not heldout.exists():
    raise FileNotFoundError(f'Missing held-out dataset: {heldout}')
for model in ['ministral', 'phi', 'qwen', 'gemma']:
    path = adapter_root / f'{model}_full_grounded'
    if not path.exists():
        raise FileNotFoundError(f'Missing adapter: {path}')
    print(model, 'adapter OK:', path)
records = [json.loads(line) for line in heldout.read_text().splitlines() if line.strip()]
print('available held-out cases:', len(records))
print('available decisions:', {d: sum(r.get('ground_truth', {}).get('decision') == d for r in records) for d in ['LEGITIMATE', 'SUSPICIOUS']})
patterns = sorted({r.get('ground_truth', {}).get('pattern') for r in records})
print('available patterns:', patterns)

In [ ]:
# Run the controlled comparison. This is the long-running cell; it resumes completed cases.
import subprocess
cmd = [
    'python', 'scripts/run_298b_fast_comparison.py',
    '--heldout', HELDOUT_DATASET,
    '--adapter-root', ADAPTER_ROOT,
    '--output-root', OUTPUT_ROOT,
    '--legitimate', str(LEGITIMATE_CASES),
    '--suspicious', str(SUSPICIOUS_CASES),
    '--max-tokens', str(MAX_TOKENS),
]
subprocess.run(cmd, check=True)

In [ ]:
# Display the final comparison table.
import pandas as pd
comparison = Path(OUTPUT_ROOT) / 'comparison' / 'model_comparison.csv'
table = pd.read_csv(comparison)
display(table.T)
print('Saved:', comparison)